# Imputaci?n del dataset de deserci?n estudiantil

Notebook preparado para el dataset `2022-03-16 dataset on student dropout .xlsx`.

La estrategia sigue el documento de imputaci?n compartido:

- Crear la variable `era` para distinguir `Pre-Tec21` y `Tec21`.
- Imputar de forma estratificada por `school + era`, y en `admission.test` tambi?n por `online.test`.
- Usar mediana para variables num?ricas y moda para variables categ?ricas.
- Conservar `No information` como categor?a cuando el faltante es informativo, agregando banderas.
- No imputar variables objetivo ni variables de fuga como `retention` o `dropout.semester`.
- Imputar variables del primer periodo solo en registros `Tec21`; en `Pre-Tec21` se conservan como faltantes estructurales.
- Exportar tres archivos: dataset completo imputado, dataset depurado para investigaci?n y reporte de an?lisis de imputaci?n.

## 1. Carga del dataset

La lectura del archivo Excel puede tardar varios minutos porque contiene 143,326 registros. Para acelerar ejecuciones futuras, el notebook crea un caché local `.pkl` después de la primera carga.

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)

DATA_PATH = Path(
    r"../data/raw/dataset-dropout.parquet"
)
OUTPUT_DIR = Path("../data/processed")

df_raw = pd.read_parquet(DATA_PATH)

print(f"shape: {df_raw.shape[0]:,} rows, {df_raw.shape[1]} features")
df_raw.head()

## 2. Limpieza inicial y variables derivadas

En el paper del dataset y en el documento de imputación se distinguen dos eras: `Pre-Tec21` para `AD14`-`AD18` y `Tec21` para `AD19`-`AD20`. También se crea `dropout` como derivada de `retention`, pero no se usa como predictor porque sería fuga de información.

In [ ]:
df = df_raw.copy()

def clean_text_columns(frame: pd.DataFrame) -> pd.DataFrame:
    frame = frame.copy()
    text_cols = frame.select_dtypes(include=["object", "string"]).columns
    for col in text_cols:
        frame[col] = (
            frame[col]
            .astype("string")
            .str.strip()
            .replace({"": pd.NA, "nan": pd.NA, "None": pd.NA, "<NA>": pd.NA})
            .astype("object")
        )
    return frame

df = clean_text_columns(df)

tec21_generations = {"AD19", "AD20"}
df["era"] = np.where(
    df["generation"].isin(tec21_generations) | df["educational.model"].eq(1),
    "Tec21",
    "Pre-Tec21",
)
df["dropout"] = np.where(df["retention"].notna(), 1 - df["retention"], np.nan)

first_period_cols = [
    "average.first.period",
    "failed.subject.first.period",
    "dropped.subject.first.period",
]
df["first_period_available"] = (
    df[first_period_cols].notna().any(axis=1).astype("int8")
)

# ── Modeling codes — on df so every copy inherits them automatically ──────────
df["era_code"] = (df["era"] == "Tec21").astype("int8")

_school_dtype = pd.CategoricalDtype(
    categories=sorted(df["school"].dropna().unique()), ordered=False
)
df["school_code"] = df["school"].astype(_school_dtype).cat.codes.astype("int16")

df[["generation", "educational.model", "era", "era_code",
    "school", "school_code", "retention", "dropout",
    "first_period_available"]].head()

## 3. Auditoría de faltantes

El documento de imputación remarca que hay que separar tres casos:

- `NaN`: faltante real.
- `No information`: falta informativa que conviene conservar como categoría en varias variables.
- `Does not apply`: valor estructural que no siempre se debe imputar, especialmente si la variable no existe para una era.

In [ ]:
SPECIAL_TOKENS = ["No information", "Does not apply", "Not defined"]


def missing_audit(frame: pd.DataFrame) -> pd.DataFrame:
    rows = []
    n = len(frame)
    for col in frame.columns:
        row = {
            "variable": col,
            "dtype": str(frame[col].dtype),
            "n_missing_nan": int(frame[col].isna().sum()),
            "pct_missing_nan": frame[col].isna().mean() * 100,
            "n_unique": int(frame[col].nunique(dropna=True)),
        }
        if frame[col].dtype == "object" or str(frame[col].dtype).startswith("string"):
            for token in SPECIAL_TOKENS:
                row[f"n_{token.lower().replace(' ', '_')}"] = int(frame[col].eq(token).sum())
        rows.append(row)
    return pd.DataFrame(rows).sort_values("pct_missing_nan", ascending=False)


audit_before = missing_audit(df)
display(audit_before.head(15))

## 4. Variables de actividades unificadas

El dataset cambió la forma de registrar actividades estudiantiles entre generaciones. Para comparar eras, se construyen:

- `activity_count_unified`: número total de actividades registradas.
- `activity_any_unified`: indicador de si participó en al menos una actividad.

Esto evita comparar columnas antiguas y nuevas como si fueran equivalentes una por una.

In [ ]:
old_activity_cols = ["physical.education", "cultural.diffusion", "student.society"]
new_activity_cols = [
    "athletic.sports",
    "art.culture",
    "student.society.leadership",
    "life.work.mentoring",
    "wellness.activities",
]
activity_cols = old_activity_cols + new_activity_cols


def to_numeric_with_tokens(series: pd.Series, extra_missing_tokens: list[str] | None = None) -> pd.Series:
    tokens = ["No information", "Does not apply"]
    if extra_missing_tokens:
        tokens.extend(extra_missing_tokens)
    cleaned = series.replace(tokens, np.nan)
    return pd.to_numeric(cleaned, errors="coerce")


activity_numeric = pd.DataFrame({col: to_numeric_with_tokens(df[col]) for col in activity_cols})
df["activity_count_unified"] = activity_numeric.sum(axis=1, min_count=1)
df["activity_any_unified"] = np.where(
    df["activity_count_unified"].isna(),
    np.nan,
    (df["activity_count_unified"] > 0).astype("int8"),
)
df["activity_missing_flag"] = df["activity_count_unified"].isna().astype("int8")

df[["era", "school", "activity_count_unified", "activity_any_unified", "activity_missing_flag"]].head()

## 5. Banderas para faltantes informativos

Para variables familiares/socioeconómicas, `No information` puede ser una señal informativa y no solo un error de captura. Por eso se conserva como categoría y se agregan banderas.

In [ ]:
keep_no_information_cols = [
    "max.degree.parents",
    "parents.exatec",
    "first.generation",
    "socioeconomic.level",
    "social.lag",
]

for col in keep_no_information_cols:
    df[f"{col}_no_information_flag"] = df[col].isna().astype("int8")
    df[col] = df[col].fillna("No information")
    df[f"{col}_no_information_flag"] = (
        df[f"{col}_no_information_flag"].eq(1) | df[col].eq("No information")
    ).astype("int8")

flag_cols = [f"{col}_no_information_flag" for col in keep_no_information_cols] + ["activity_missing_flag"]
df[keep_no_information_cols + flag_cols].head()

## 6. Funciones de imputación estratificada

Las funciones intentan imputar primero con el grupo recomendado. Si el grupo no tiene valores observados, usan grupos de respaldo y finalmente la mediana/moda global. Esto evita que queden nulos cuando un estrato `school + era` es pequeño.

In [ ]:
imputation_log = []


def _groups_as_list(group_cols):
    return list(group_cols) if isinstance(group_cols, (list, tuple)) else [group_cols]


def add_missing_flag(frame: pd.DataFrame, col: str, suffix: str = "missing_flag") -> None:
    frame[f"{col}_{suffix}"] = frame[col].isna().astype("int8")


def groupwise_median_impute(
    frame: pd.DataFrame,
    col: str,
    group_cols: list[str],
    fallback_groups: list[list[str]] | None = None,
    round_result: bool = False,
    clip_min: float | None = None,
    clip_max: float | None = None,
) -> pd.Series:
    fallback_groups = fallback_groups or []
    before = int(frame[col].isna().sum())
    result = frame[col].copy()

    for groups in [group_cols] + fallback_groups:
        groups = _groups_as_list(groups)
        fill_values = frame.groupby(groups, dropna=False)[col].transform("median")
        result = result.fillna(fill_values)

    result = result.fillna(frame[col].median())

    if clip_min is not None or clip_max is not None:
        result = result.clip(lower=clip_min, upper=clip_max)
    if round_result:
        result = result.round()

    after = int(result.isna().sum())
    imputation_log.append(
        {
            "variable": col,
            "method": "mediana estratificada",
            "primary_group": " + ".join(group_cols),
            "missing_before": before,
            "missing_after": after,
        }
    )
    return result


def _mode_or_na(values: pd.Series):
    clean = values.dropna()
    if clean.empty:
        return pd.NA
    return clean.mode().iloc[0]


def groupwise_mode_impute(
    frame: pd.DataFrame,
    col: str,
    group_cols: list[str],
    fallback_groups: list[list[str]] | None = None,
    final_value: str | int | float | None = None,
) -> pd.Series:
    fallback_groups = fallback_groups or []
    before = int(frame[col].isna().sum())
    result = frame[col].copy()

    for groups in [group_cols] + fallback_groups:
        groups = _groups_as_list(groups)
        fill_values = frame.groupby(groups, dropna=False)[col].transform(_mode_or_na)
        result = result.fillna(fill_values)

    if final_value is not None:
        result = result.fillna(final_value)
    else:
        result = result.fillna(_mode_or_na(frame[col]))

    after = int(result.isna().sum())
    imputation_log.append(
        {
            "variable": col,
            "method": "moda estratificada",
            "primary_group": " + ".join(group_cols),
            "missing_before": before,
            "missing_after": after,
        }
    )
    return result

## 7. Preparación de variables numéricas/categóricas a imputar

Se convierten a numéricas solo las variables que conceptualmente son puntajes o porcentajes. Los tokens textuales se vuelven `NaN` para que puedan imputarse con mediana cuando corresponde.

In [ ]:
numeric_impute_cols = [
    "admission.test",
    "admission.rubric",
    "general.math.eval",
    "scholarship.perc",
    "loan.perc",
    "activity_count_unified",
]

def imputable_missing_mask(series: pd.Series) -> pd.Series:
    mask = series.isna()
    if series.dtype == "object" or str(series.dtype).startswith("string"):
        token_mask = series.astype("string").str.strip().isin(["No information", "Does not apply"])
        mask = mask | token_mask.fillna(False)
    return mask

for col in numeric_impute_cols:
    df[f"{col}_missing_flag"] = imputable_missing_mask(df[col]).astype("int8")

df["admission.test"] = to_numeric_with_tokens(df["admission.test"])
df["general.math.eval"] = to_numeric_with_tokens(df["general.math.eval"])
df["admission.rubric"] = pd.to_numeric(df["admission.rubric"], errors="coerce")
df["scholarship.perc"] = pd.to_numeric(df["scholarship.perc"], errors="coerce")
df["loan.perc"] = pd.to_numeric(df["loan.perc"], errors="coerce")
df["activity_count_unified"] = pd.to_numeric(df["activity_count_unified"], errors="coerce")

# Si no hay apoyo financiero, beca y préstamo deben ser 0 antes de usar medianas.
no_scholarship = df["scholarship.type"].eq("No scholarship")
df.loc[no_scholarship & df["scholarship.perc"].isna(), "scholarship.perc"] = 0
df.loc[no_scholarship & df["loan.perc"].isna(), "loan.perc"] = 0

categorical_impute_cols = ["school.cost", "scholarship.type", "activity_any_unified"]
for col in categorical_impute_cols:
    add_missing_flag(df, col)

df[numeric_impute_cols + [f"{c}_missing_flag" for c in numeric_impute_cols]].head()

## 8. Imputaci?n del dataset completo

Esta secci?n imputa una copia completa del dataset original. Las columnas originales se conservan y las variables derivadas/banderas quedan al final para mantener trazabilidad.

In [ ]:
df_before_imputation = df.copy()
df_imputed = df.copy()

# Variables num?ricas principales: mediana estratificada.
df_imputed["admission.test"] = groupwise_median_impute(
    df_imputed,
    "admission.test",
    ["school", "era", "online.test"],
    fallback_groups=[["school", "era"], ["era", "online.test"], ["era"]],
)
df_imputed["admission.rubric"] = groupwise_median_impute(
    df_imputed,
    "admission.rubric",
    ["school", "era"],
    fallback_groups=[["school"], ["era"]],
)
df_imputed["general.math.eval"] = groupwise_median_impute(
    df_imputed,
    "general.math.eval",
    ["school", "era"],
    fallback_groups=[["school"], ["era"]],
)
df_imputed["scholarship.perc"] = groupwise_median_impute(
    df_imputed,
    "scholarship.perc",
    ["school", "era", "scholarship.type"],
    fallback_groups=[["school", "era"], ["era"]],
    clip_min=0,
    clip_max=1,
)
df_imputed["loan.perc"] = groupwise_median_impute(
    df_imputed,
    "loan.perc",
    ["school", "era", "scholarship.type"],
    fallback_groups=[["school", "era"], ["era"]],
    clip_min=0,
    clip_max=1,
)
df_imputed["activity_count_unified"] = groupwise_median_impute(
    df_imputed,
    "activity_count_unified",
    ["school", "era"],
    fallback_groups=[["school"], ["era"]],
    round_result=True,
    clip_min=0,
)

# Variables categ?ricas o binarias: moda estratificada.
df_imputed["activity_any_unified"] = groupwise_mode_impute(
    df_imputed,
    "activity_any_unified",
    ["school", "era"],
    fallback_groups=[["school"], ["era"]],
    final_value=0,
).astype("int8")

df_imputed["school.cost"] = groupwise_mode_impute(
    df_imputed,
    "school.cost",
    ["school", "era"],
    fallback_groups=[["school"], ["era"]],
    final_value="Not defined",
)

df_imputed.loc[
    df_imputed["scholarship.type"].isna()
    & df_imputed["scholarship.perc"].fillna(0).eq(0)
    & df_imputed["loan.perc"].fillna(0).eq(0),
    "scholarship.type",
] = "No scholarship"
df_imputed["scholarship.type"] = groupwise_mode_impute(
    df_imputed,
    "scholarship.type",
    ["school", "era"],
    fallback_groups=[["school"], ["era"]],
    final_value="No scholarship",
)

# Variables del primer periodo: solo se imputan dentro de Tec21.
tec21_mask = df_imputed["era"].eq("Tec21")
df_imputed["first_period_structural_missing_flag"] = (~tec21_mask).astype("int8")

first_period_specs = {
    "average.first.period": {"clip_min": 0, "clip_max": 100, "round_result": False},
    "failed.subject.first.period": {"clip_min": 0, "clip_max": None, "round_result": True},
    "dropped.subject.first.period": {"clip_min": 0, "clip_max": None, "round_result": True},
}

for col, spec in first_period_specs.items():
    df_imputed[f"{col}_missing_flag"] = df_imputed[col].isna().astype("int8")
    df_imputed[col] = pd.to_numeric(df_imputed[col], errors="coerce")

    tec21_subset = df_imputed.loc[tec21_mask].copy()
    tec21_subset[col] = groupwise_median_impute(
        tec21_subset,
        col,
        ["school", "generation"],
        fallback_groups=[["school"], ["generation"]],
        round_result=spec["round_result"],
        clip_min=spec["clip_min"],
        clip_max=spec["clip_max"],
    )
    df_imputed.loc[tec21_subset.index, col] = tec21_subset[col]

# Orden: columnas originales primero; variables derivadas y banderas al final.
original_cols = list(df_raw.columns)
extra_cols = [col for col in df_imputed.columns if col not in original_cols]
df_imputed_final = df_imputed[original_cols + extra_cols].copy()

print(f"Dataset imputado completo: {df_imputed_final.shape[0]:,} filas | {df_imputed_final.shape[1]:,} columnas")
display(df_imputed_final.head())

## 9. Dataset depurado para investigaci?n

Con base en los documentos compartidos, se conserva un conjunto de variables ?tiles para estudiar la deserci?n por escuela y por era/modelo. Se descartan identificadores, variables redundantes, columnas demasiado granulares, variables espec?ficas de actividades ya unificadas y resultados que generar?an fuga de informaci?n.

In [ ]:
import json

# ── Tec21-only model inputs: NaN → 0, then × era_code wipes Pre-Tec21 rows ───
# Result: Pre-Tec21 → 0.0 (structural zero, not an imputed value);
#         Tec21     → imputed/observed value.
# first_period_structural_missing_flag distinguishes the two cases in the model.
for col in first_period_cols:
    df_imputed_final[f"{col}_model"] = (
        df_imputed_final[col].fillna(0.0) * df_imputed_final["era_code"]
    ).astype("float32")

# ── Column selection metadata ─────────────────────────────────────────────────
selected_column_metadata = [
    {"column": "generation",        "role": "contexto",           "scope": "comparativo", "reason": "Define cohorte y permite separar Pre-Tec21 vs Tec21."},
    {"column": "educational.model", "role": "contexto",           "scope": "comparativo", "reason": "Identifica modelo educativo; no se imputa."},
    {"column": "era",               "role": "contexto derivado",  "scope": "comparativo", "reason": "Variable clave derivada para comparar Pre-Tec21 y Tec21."},
    {"column": "era_code",          "role": "grupo derivado",     "scope": "comparativo", "reason": "Indicador binario Pre-Tec21=0 / Tec21=1; requerido por sklearn y PyMC."},
    {"column": "school",            "role": "estratificación",    "scope": "comparativo", "reason": "Base de la heterogeneidad estructural por escuela."},
    {"column": "school_code",       "role": "grupo",              "scope": "comparativo", "reason": "Índice entero de escuela para efectos aleatorios en PyMC."},
    {"column": "region",            "role": "contexto",           "scope": "comparativo", "reason": "Control institucional/geográfico recomendado junto con escuela y era."},
    {"column": "level",             "role": "contexto",           "scope": "comparativo", "reason": "Distingue High School y Undergraduate."},
    {"column": "retention",         "role": "target",             "scope": "modelado",    "reason": "Variable objetivo; se conserva sin imputar."},
    {"column": "dropout",           "role": "target derivado",    "scope": "modelado",    "reason": "Derivada de retention para análisis de abandono; no se usa como predictor si retention es target."},
    {"column": "gender",            "role": "predictor",          "scope": "comparativo", "reason": "Variable demográfica de control."},
    {"column": "age",               "role": "predictor",          "scope": "comparativo", "reason": "Variable demográfica de control."},
    {"column": "foreign",           "role": "predictor",          "scope": "comparativo", "reason": "Contexto de procedencia del estudiante."},
    {"column": "tec.no.tec",        "role": "predictor",          "scope": "comparativo", "reason": "Indica si proviene del sistema Tec."},
    {"column": "PNA",               "role": "predictor académico","scope": "comparativo", "reason": "Rendimiento previo; aproxima preparación académica."},
    {"column": "admission.test",    "role": "predictor imputado", "scope": "comparativo", "reason": "Preparación académica de ingreso; mediana por school + era + online.test."},
    {"column": "online.test",       "role": "predictor",          "scope": "comparativo", "reason": "Control del formato de prueba de admisión."},
    {"column": "english.evaluation","role": "predictor académico","scope": "comparativo", "reason": "Nivel de inglés previo."},
    {"column": "admission.rubric",  "role": "predictor imputado", "scope": "comparativo", "reason": "Resume perfil de admisión; mediana por school + era."},
    {"column": "general.math.eval", "role": "predictor imputado", "scope": "comparativo", "reason": "Preparación matemática previa."},
    {"column": "FTE",               "role": "predictor académico","scope": "comparativo", "reason": "Carga/estatus de tiempo completo."},
    {"column": "scholarship.perc",  "role": "predictor imputado", "scope": "comparativo", "reason": "Apoyo económico directo; factor financiero relevante."},
    {"column": "loan.perc",         "role": "predictor imputado", "scope": "comparativo", "reason": "Complementa análisis de apoyo financiero."},
    {"column": "school.cost",       "role": "predictor imputado", "scope": "comparativo", "reason": "Aproxima contexto económico previo; moda por school + era."},
    {"column": "scholarship.type",  "role": "predictor imputado", "scope": "comparativo", "reason": "Tipo de apoyo financiero."},
    {"column": "max.degree.parents","role": "predictor familiar", "scope": "comparativo", "reason": "Aproxima capital cultural familiar; conserva No information."},
    {"column": "parents.exatec",    "role": "predictor familiar", "scope": "comparativo", "reason": "Capital social/familiar vinculado al Tec; conserva No information."},
    {"column": "first.generation",  "role": "predictor familiar", "scope": "comparativo", "reason": "Clave para estudiar desigualdad estructural; conserva No information."},
    {"column": "socioeconomic.level","role": "predictor socioeconómico","scope": "comparativo","reason": "Se conserva como categoría con No information por alto faltante informativo."},
    {"column": "social.lag",        "role": "predictor socioeconómico","scope": "comparativo","reason": "Se conserva como categoría con No information por alto faltante informativo."},
    {"column": "activity_count_unified","role": "predictor imputado","scope": "comparativo","reason": "Integra participación estudiantil entre modelos educativos."},
    {"column": "activity_any_unified",  "role": "predictor imputado","scope": "comparativo","reason": "Indicador interpretable de participación en actividades."},
    # Tec21-only — original (NaN for Pre-Tec21; use for EDA/analysis)
    {"column": "average.first.period",              "role": "predictor Tec21-only", "scope": "Tec21", "reason": "Desempeño temprano; NaN estructural para Pre-Tec21."},
    {"column": "failed.subject.first.period",       "role": "predictor Tec21-only", "scope": "Tec21", "reason": "Riesgo académico temprano; NaN estructural para Pre-Tec21."},
    {"column": "dropped.subject.first.period",      "role": "predictor Tec21-only", "scope": "Tec21", "reason": "Bajas tempranas; NaN estructural para Pre-Tec21."},
    # Tec21-only — model-ready (0 for Pre-Tec21 via × era_code; use for sklearn/PyMC)
    {"column": "average.first.period_model",        "role": "predictor Tec21-only modelo", "scope": "Tec21", "reason": "Promedio primer periodo; 0 para Pre-Tec21, valor imputado/observado para Tec21."},
    {"column": "failed.subject.first.period_model", "role": "predictor Tec21-only modelo", "scope": "Tec21", "reason": "Materias reprobadas primer periodo; 0 para Pre-Tec21."},
    {"column": "dropped.subject.first.period_model","role": "predictor Tec21-only modelo", "scope": "Tec21", "reason": "Bajas primer periodo; 0 para Pre-Tec21."},
    # Flags
    {"column": "first_period_available",                  "role": "bandera", "scope": "Tec21",       "reason": "Disponibilidad real de variables del primer periodo."},
    {"column": "first_period_structural_missing_flag",    "role": "bandera", "scope": "comparativo", "reason": "Marca faltante estructural de primer periodo en Pre-Tec21."},
]

flag_columns = [
    col for col in df_imputed_final.columns
    if col.endswith("_missing_flag")
    or col.endswith("_no_information_flag")
    or col == "activity_missing_flag"
]

base_research_columns = [item["column"] for item in selected_column_metadata]
research_columns = [
    col for col in base_research_columns + flag_columns
    if col in df_imputed_final.columns
]
research_columns = list(dict.fromkeys(research_columns))

discarded_column_metadata = [
    {"column": "student.id",               "reason": "Identificador anonimizado; puede inducir sobreajuste."},
    {"column": "id.school.origin",         "reason": "Demasiadas categorías; baja interpretabilidad."},
    {"column": "father.education.complete","reason": "Granular y redundante con max.degree.parents."},
    {"column": "mother.education.complete","reason": "Granular y redundante con max.degree.parents."},
    {"column": "father.education.summary", "reason": "Redundante si se usa max.degree.parents."},
    {"column": "mother.education.summary", "reason": "Redundante si se usa max.degree.parents."},
    {"column": "father.exatec",            "reason": "Redundante con parents.exatec."},
    {"column": "mother.exatec",            "reason": "Redundante con parents.exatec."},
    {"column": "total.scholarship.loan",   "reason": "Redundante con scholarship.perc y loan.perc."},
    {"column": "program",                  "reason": "Demasiado granular; se prioriza school."},
    {"column": "zone.type",                "reason": "Alta proporción de No information; menor aporte frente a school.cost y socioeconomic.level."},
    {"column": "dropout.semester",         "reason": "Resultado posterior del abandono; fuga de información como predictor."},
    {"column": "physical.education",       "reason": "Actividad específica de generaciones anteriores; reemplazada por activity_count_unified."},
    {"column": "cultural.diffusion",       "reason": "Actividad específica de generaciones anteriores; reemplazada por variables unificadas."},
    {"column": "student.society",          "reason": "Actividad específica de generaciones anteriores; reemplazada por variables unificadas."},
    {"column": "total.life.activities",    "reason": "No comparable entre eras; se usa actividad unificada."},
    {"column": "athletic.sports",          "reason": "Actividad específica de Tec21; reemplazada por variables unificadas."},
    {"column": "art.culture",              "reason": "Actividad específica de Tec21; reemplazada por variables unificadas."},
    {"column": "student.society.leadership","reason": "Actividad específica de Tec21; reemplazada por variables unificadas."},
    {"column": "life.work.mentoring",      "reason": "Actividad específica de Tec21; reemplazada por variables unificadas."},
    {"column": "wellness.activities",      "reason": "Actividad específica de Tec21; reemplazada por variables unificadas."},
]

implicit_discarded = [
    col for col in df_imputed_final.columns
    if col not in research_columns and col not in [d["column"] for d in discarded_column_metadata]
]
for col in implicit_discarded:
    discarded_column_metadata.append({
        "column": col,
        "reason": "Columna auxiliar fuera del dataset depurado.",
    })

df_research = df_imputed_final[research_columns].copy()

# ── School lookup ─────────────────────────────────────────────────────────────
school_lookup = (
    df_research[["school", "school_code"]]
    .drop_duplicates()
    .sort_values("school_code")
    .reset_index(drop=True)
)

# ── Feature manifest ──────────────────────────────────────────────────────────
FEATURE_MANIFEST = {
    "target": "dropout",
    "numeric_common": [
        "age", "PNA", "admission.test", "english.evaluation",
        "admission.rubric", "general.math.eval", "FTE",
        "scholarship.perc", "loan.perc", "activity_count_unified",
    ],
    "numeric_tec21_model": [
        "average.first.period_model",
        "failed.subject.first.period_model",
        "dropped.subject.first.period_model",
    ],
    "categorical_common": [
        "gender", "foreign", "tec.no.tec", "online.test",
        "school.cost", "scholarship.type",
        "max.degree.parents", "parents.exatec",
        "first.generation", "socioeconomic.level", "social.lag",
    ],
    "missing_flags": [c for c in flag_columns if c in df_research.columns],
    "group_vars": {
        "era_code":   "int8  — 0=Pre-Tec21, 1=Tec21",
        "school_code":"int16 — alphabetical index; see school_code_lookup.csv",
    },
    "leakage_vars": ["retention", "dropout"],
}

print(f"Dataset depurado: {df_research.shape[0]:,} filas | {df_research.shape[1]} columnas")
print(f"  Columnas descartadas: {len([c for c in df_imputed_final.columns if c not in research_columns])}")
print(f"  Schools: {len(school_lookup)}  |  Era split: {df_research['era'].value_counts().to_dict()}")
display(df_research.head())

## 10. Reporte y exportación de 3 outputs

Se generan tres archivos en `outputs_imputacion`:

1. Dataset completo imputado.
2. Dataset depurado para investigaci?n.
3. Reporte Excel con estad?sticas antes/despu?s, columnas imputadas/no imputadas, columnas conservadas/descartadas y porcentajes de vac?os por escuela y modelo.

In [ ]:
def missing_profile(frame: pd.DataFrame, columns: list[str] | None = None, dataset_name: str = "dataset") -> pd.DataFrame:
    columns = columns or list(frame.columns)
    rows = []
    n_rows = len(frame)
    for col in columns:
        if col not in frame.columns:
            continue
        series = frame[col]
        row = {
            "dataset": dataset_name,
            "column": col,
            "dtype": str(series.dtype),
            "rows": n_rows,
            "missing_count": int(series.isna().sum()),
            "missing_pct": round(float(series.isna().mean() * 100), 4),
            "unique_non_null": int(series.nunique(dropna=True)),
        }
        if series.dtype == "object" or str(series.dtype).startswith("string"):
            row["no_information_count"] = int(series.eq("No information").sum())
            row["does_not_apply_count"] = int(series.eq("Does not apply").sum())
            row["not_defined_count"] = int(series.eq("Not defined").sum())
        rows.append(row)
    return pd.DataFrame(rows)


def before_after_profile(before: pd.DataFrame, after: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    before_prof = missing_profile(before, columns, "antes").rename(
        columns={
            "missing_count": "missing_before",
            "missing_pct": "missing_pct_before",
            "unique_non_null": "unique_before",
        }
    )
    after_prof = missing_profile(after, columns, "despues").rename(
        columns={
            "missing_count": "missing_after",
            "missing_pct": "missing_pct_after",
            "unique_non_null": "unique_after",
        }
    )
    keep_before = ["column", "dtype", "rows", "missing_before", "missing_pct_before", "unique_before"]
    keep_after = ["column", "missing_after", "missing_pct_after", "unique_after"]
    merged = before_prof[keep_before].merge(after_prof[keep_after], on="column", how="outer")
    merged["missing_reduced_count"] = merged["missing_before"].fillna(0) - merged["missing_after"].fillna(0)
    merged["was_imputed"] = merged["column"].isin(imputed_columns)
    merged["in_research_dataset"] = merged["column"].isin(research_columns)
    return merged.sort_values(["was_imputed", "missing_before"], ascending=[False, False])


def segmented_missing_report(
    before: pd.DataFrame,
    after: pd.DataFrame,
    segment_cols: list[str],
    variables: list[str],
    dataset_name: str,
) -> pd.DataFrame:
    rows = []
    variables = [col for col in variables if col in before.columns and col in after.columns]
    grouped_before = before.groupby(segment_cols, dropna=False)
    grouped_after = after.groupby(segment_cols, dropna=False)
    for keys, before_group in grouped_before:
        if not isinstance(keys, tuple):
            keys = (keys,)
        try:
            after_group = grouped_after.get_group(keys[0] if len(keys) == 1 else keys)
        except KeyError:
            after_group = after.iloc[0:0]
        base = {segment_cols[i]: keys[i] for i in range(len(segment_cols))}
        for var in variables:
            n = len(before_group)
            before_missing = int(before_group[var].isna().sum())
            after_missing = int(after_group[var].isna().sum()) if len(after_group) else 0
            rows.append(
                {
                    "dataset": dataset_name,
                    **base,
                    "variable": var,
                    "rows": n,
                    "missing_before": before_missing,
                    "missing_pct_before": round(before_missing / n * 100, 4) if n else 0,
                    "missing_after": after_missing,
                    "missing_pct_after": round(after_missing / n * 100, 4) if n else 0,
                }
            )
    return pd.DataFrame(rows)

In [ ]:
key_imputed_cols = [
    "admission.test",
    "admission.rubric",
    "general.math.eval",
    "scholarship.perc",
    "loan.perc",
    "activity_count_unified",
    "activity_any_unified",
    "school.cost",
    "scholarship.type",
]
key_tec21_cols = key_imputed_cols + first_period_cols

validation = pd.DataFrame(
    {
        "missing_after_total": df_imputed_final[key_tec21_cols].isna().sum(),
        "missing_after_tec21": df_imputed_final.loc[df_imputed_final["era"].eq("Tec21"), key_tec21_cols].isna().sum(),
        "missing_after_pretec21": df_imputed_final.loc[df_imputed_final["era"].eq("Pre-Tec21"), key_tec21_cols].isna().sum(),
    }
)

imputed_columns = [
    "admission.test",
    "admission.rubric",
    "general.math.eval",
    "scholarship.perc",
    "loan.perc",
    "activity_count_unified",
    "activity_any_unified",
    "school.cost",
    "scholarship.type",
    "average.first.period",
    "failed.subject.first.period",
    "dropped.subject.first.period",
]

not_imputed_reason = {
    "retention": "Variable objetivo; no se imputa para evitar contaminar el modelo.",
    "dropout": "Variable objetivo derivada de retention; no se imputa como predictor.",
    "dropout.semester": "Resultado posterior del abandono; se excluye del dataset depurado por fuga de informaci?n.",
    "generation": "Cohorte clave; si faltara no debe inventarse.",
    "educational.model": "Modelo educativo clave; no debe inventarse.",
    "era": "Variable derivada de generation/modelo; no se imputa.",
    "school": "Estrato central de heterogeneidad; si faltara no conviene asignarlo artificialmente.",
    "max.degree.parents": "Se conserva No information como categor?a informativa y se agrega bandera.",
    "parents.exatec": "Se conserva No information como categor?a informativa y se agrega bandera.",
    "first.generation": "Se conserva No information como categor?a informativa y se agrega bandera.",
    "socioeconomic.level": "Alto No information; se conserva como categor?a informativa.",
    "social.lag": "Alto No information; se conserva como categor?a informativa.",
}
for item in discarded_column_metadata:
    not_imputed_reason.setdefault(item["column"], item["reason"])

not_imputed_columns = [col for col in df_imputed_final.columns if col not in imputed_columns]
not_imputed_report = pd.DataFrame(
    [
        {
            "column": col,
            "in_research_dataset": col in research_columns,
            "reason": not_imputed_reason.get(col, "Sin imputaci?n directa: sin faltantes imputables, variable de control, bandera o derivada de trazabilidad."),
        }
        for col in not_imputed_columns
    ]
)

In [ ]:
complete_profile = before_after_profile(df_before_imputation, df_imputed_final, list(df_imputed_final.columns))
research_profile = before_after_profile(
    df_before_imputation.reindex(columns=research_columns),
    df_research,
    research_columns,
)

summary_datasets = pd.DataFrame(
    [
        {
            "dataset": "completo_imputado",
            "rows": df_imputed_final.shape[0],
            "columns": df_imputed_final.shape[1],
            "missing_cells": int(df_imputed_final.isna().sum().sum()),
            "missing_cells_pct": round(float(df_imputed_final.isna().sum().sum() / df_imputed_final.size * 100), 4),
        },
        {
            "dataset": "investigacion_depurado",
            "rows": df_research.shape[0],
            "columns": df_research.shape[1],
            "missing_cells": int(df_research.isna().sum().sum()),
            "missing_cells_pct": round(float(df_research.isna().sum().sum() / df_research.size * 100), 4),
        },
    ]
)

selected_columns_report = pd.DataFrame(selected_column_metadata)
selected_columns_report = pd.concat(
    [
        selected_columns_report,
        pd.DataFrame(
            [
                {"column": col, "role": "bandera", "scope": "trazabilidad", "reason": "Bandera de faltante/imputaci?n conservada para modelar missingness no aleatorio."}
                for col in flag_columns
                if col in research_columns and col not in selected_columns_report["column"].tolist()
            ]
        ),
    ],
    ignore_index=True,
)

discarded_columns_report = pd.DataFrame(discarded_column_metadata)
discarded_columns_report = discarded_columns_report[discarded_columns_report["column"].isin(df_imputed_final.columns)]
discarded_columns_report = discarded_columns_report.drop_duplicates("column").sort_values("column")

segment_variables = list(dict.fromkeys(imputed_columns + keep_no_information_cols + ["PNA", "retention", "dropout"]))
missing_by_school_model = segmented_missing_report(
    df_before_imputation,
    df_imputed_final,
    ["school", "era", "educational.model"],
    segment_variables,
    "completo_imputado",
)
missing_by_model = segmented_missing_report(
    df_before_imputation,
    df_imputed_final,
    ["era", "educational.model"],
    segment_variables,
    "completo_imputado",
)

In [ ]:
import json

output_complete_path  = OUTPUT_DIR / "dataset_student_dropout_imputado_completo.csv"
output_research_path  = OUTPUT_DIR / "dataset_student_dropout_imputado_investigacion.csv"
output_report_path    = OUTPUT_DIR / "reporte_analisis_imputacion.xlsx"
output_lookup_path    = OUTPUT_DIR / "school_code_lookup.csv"
output_manifest_path  = OUTPUT_DIR / "feature_manifest.json"

known_outputs = [
    "dataset_student_dropout_imputado_completo.csv",
    "dataset_student_dropout_imputado_investigacion.csv",
    "reporte_analisis_imputacion.xlsx",
    "school_code_lookup.csv",
    "feature_manifest.json",
    "dropout_imputado_comparativo_pretec21_vs_tec21.csv",
    "dropout_imputado_tec21_only.csv",
    "reporte_imputacion.csv",
    "validacion_faltantes_post_imputacion.csv",
    "auditoria_faltantes_antes.csv",
]
for filename in known_outputs:
    candidate = OUTPUT_DIR / filename
    if candidate.exists():
        candidate.unlink()

# ── Tabular outputs ───────────────────────────────────────────────────────────
df_imputed_final.to_csv(output_complete_path, index=False, encoding="utf-8-sig")
df_research.to_csv(output_research_path,       index=False, encoding="utf-8-sig")

df_imputed_final.to_parquet(OUTPUT_DIR / "dataset_student_dropout_imputado_completo.parquet",      index=False)
df_research.to_parquet(     OUTPUT_DIR / "dataset_student_dropout_imputado_investigacion.parquet", index=False)

# ── Modeling metadata ─────────────────────────────────────────────────────────
school_lookup.to_csv(output_lookup_path, index=False)

with open(output_manifest_path, "w", encoding="utf-8") as f:
    json.dump(FEATURE_MANIFEST, f, indent=2, ensure_ascii=False)

# ── Excel report (unchanged sheets) ──────────────────────────────────────────
summary_datasets.to_parquet(OUTPUT_DIR / "reporte_resumen_datasets.parquet",     index=False)
complete_profile.to_parquet(OUTPUT_DIR / "reporte_faltantes_completo.parquet",   index=False)
research_profile.to_parquet(OUTPUT_DIR / "reporte_faltantes_investigacion.parquet", index=False)

with pd.ExcelWriter(output_report_path, engine="openpyxl") as writer:
    summary_datasets.to_excel(      writer, sheet_name="resumen_datasets",       index=False)
    pd.DataFrame(imputation_log).to_excel(writer, sheet_name="columnas_imputadas", index=False)
    not_imputed_report.to_excel(    writer, sheet_name="columnas_no_imputadas",   index=False)
    selected_columns_report.to_excel(writer,sheet_name="columnas_conservadas",   index=False)
    discarded_columns_report.to_excel(writer,sheet_name="columnas_descartadas",  index=False)
    complete_profile.to_excel(      writer, sheet_name="faltantes_completo",      index=False)
    research_profile.to_excel(      writer, sheet_name="faltantes_investigacion", index=False)
    missing_by_school_model.to_excel(writer,sheet_name="vacios_escuela_modelo",  index=False)
    missing_by_model.to_excel(      writer, sheet_name="vacios_modelo",           index=False)
    validation.to_excel(            writer, sheet_name="validacion_clave",        index=True)
    school_lookup.to_excel(         writer, sheet_name="school_code_lookup",      index=False)

print("Outputs generados:")
print(f"  {output_complete_path}")
print(f"  {output_research_path}")
print(f"  {output_report_path}")
print(f"  {output_lookup_path}")
print(f"  {output_manifest_path}")
display(summary_datasets)

## 11. Siguiente paso para modelado

Para un modelo predictivo final, evita ajustar imputaciones usando todo el dataset antes de separar entrenamiento/prueba. Lo m?s correcto es llevar estas reglas a un `Pipeline` o ajustar las medianas/modas solo en entrenamiento y aplicarlas al conjunto de prueba. Para esta auditor?a/pre-entrega, este notebook deja tres entregables trazables: el dataset completo imputado, el dataset depurado para investigaci?n y un reporte de an?lisis de imputaci?n.